# TerraClimate Extended — All 14 Variables

Extension of the original TerraClimate notebook that saves ALL available climate variables
instead of just `srad` and `vap`. This produces a richer feature set for the prediction model.

**Variables saved:** aet, def, pet, ppt, q, soil, srad, swe, tmax, tmin, vap, ws, vpd, pdsi

**Output:** `TerraClimate_all_vars.tiff` (14 bands)

In [ ]:
!pip install rioxarray pystac-client planetary-computer zarr adlfs --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import xarray as xr
import pandas as pd
import rioxarray as rio
import rasterio
import pystac_client
import planetary_computer

print('Libraries loaded')

In [ ]:
# Access TerraClimate via Microsoft Planetary Computer
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace
)

collection = catalog.get_collection('terraclimate')
asset = collection.assets['zarr-abfs']

ds = xr.open_dataset(asset.href, **asset.extra_fields['xarray:open_kwargs'])
ds = ds.drop('crs', dim=None)
print('Dataset variables:', list(ds.data_vars))

In [ ]:
# Trim to 2017-2019 and southeastern Australia bounding box
ds = ds.sel(time=slice('2017-11-01', '2019-11-01'))

min_lon, max_lon = 139.94, 151.48
min_lat, max_lat = -39.74, -30.92

mask_lon = (ds.lon >= min_lon) & (ds.lon <= max_lon)
mask_lat = (ds.lat >= min_lat) & (ds.lat <= max_lat)
ds = ds.where(mask_lon & mask_lat, drop=True)

print('Cropped dataset:', ds)
print(f'Memory: {ds.nbytes / 1024**2:.0f} MB')

In [ ]:
# Persist in memory and compute median across time
ds = ds.persist()
median = ds.median(dim='time', skipna=True).compute()
print('Median computed')

In [ ]:
# All 14 TerraClimate variables to save
ALL_VARS = ['aet', 'def', 'pet', 'ppt', 'q', 'soil',
            'srad', 'swe', 'tmax', 'tmin', 'vap', 'ws', 'vpd', 'pdsi']

# Keep only variables that exist in the dataset
available = [v for v in ALL_VARS if v in median.data_vars]
print(f'Variables to save ({len(available)}):', available)

height = median.dims['lat']
width  = median.dims['lon']

gt = rasterio.transform.from_bounds(min_lon, min_lat, max_lon, max_lat, width, height)

filename = 'TerraClimate_all_vars.tiff'

with rasterio.open(
    filename, 'w', driver='GTiff',
    width=width, height=height,
    crs='epsg:4326', transform=gt,
    count=len(available),
    compress='lzw', dtype='float64'
) as dst:
    for i, var in enumerate(available, start=1):
        dst.write(median[var].values, i)
        dst.update_tags(i, name=var)
        print(f'  Band {i}: {var}')

print(f'\nSaved: {filename}')
print(f'Bands: {len(available)}')

In [ ]:
# Verify: also save the original 2-band file for backward compatibility
with rasterio.open(
    'TerraClimate_output.tiff', 'w', driver='GTiff',
    width=width, height=height,
    crs='epsg:4326', transform=gt,
    count=2, compress='lzw', dtype='float64'
) as dst:
    dst.write(median.srad.values, 1)
    dst.write(median.vap.values, 2)
print('Also saved: TerraClimate_output.tiff (srad, vap)')

## Next step

Now run **`Improved_Biodiversity_Model.ipynb`** to train the optimized model and generate a submission.